# RQ1 & RQ3 — Cloud-native vs. Traditional Geospatial QueriesAnalysis of single-machine query benchmarks comparing Local (Python),PostGIS and DuckDB across three workload types and two dataset sizes.

In [1]:
from __future__ import annotations

import platform
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import yaml

from src.analysis.loading import (
    enrich_costs, enrich_samples, load_cached, load_costs, load_experiments,
    load_metadata, load_samples_cached, open_cache,
)
from src.analysis.tables import (
    build_consistency_table, build_cost_summary, build_cross_pass_table,
    build_descriptive_table, build_pairwise_table,
    build_rq1_ranking, build_scaling_table,
    compact_descriptive_table, format_consistency_table,
    format_cost_table, format_ranking_table,
    split_pairwise_effects, split_pairwise_significance,
)
from src.analysis.validation import run_all_validations
from src.plotting.style import StyleConfig
from src.plotting.charts import (
    plot_cost_bars, plot_cost_breakdown, plot_cost_performance,
    plot_coverage_heatmap, plot_median_bars, plot_violin,
    plot_cpu_breakdown, plot_network_io, plot_cv_iqr,
    plot_cross_workload, plot_size_scaling, plot_ranking_stability,
)

In [2]:
# RUN_ID = "2026-05-26-L6OCV2"
# RUN_ID = "2026-05-24-HBJYYT"
RUN_ID = "2026-05-26-OY0CN3"
BLOB_STORAGE_ACCOUNT_NAME = "doppabs"
BENCHMARK_CONTAINER = "benchmarks"
METADATA_CONTAINER = "metadata"
BENCHMARK_METADATA_BLOB_NAME = "benchmark_metadata.parquet"
BENCHMARKS_YML_PATH = Path("../doppa/benchmarks.yml")

FIGURES_DIR = Path("figures")
TABLES_DIR = Path("tables")
FIGURES_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(exist_ok=True)

PRIMARY_METRICS = ["elapsed_time", "network_bytes_received", "network_bytes_sent"]
AUXILIARY_METRICS = ["cpu_time_user_seconds", "cpu_time_system_seconds"]

ITERATION_CEILINGS = {
    "point-in-polygon-lookup": 2500,
    "knn-search": 4000,
    "bbox-filtering": 900,
    "national-scale-spatial-join": 5,
}
WORKLOAD_TYPES = sorted(ITERATION_CEILINGS.keys(), key=len, reverse=True)

RQ1_WORKLOADS = ["point-in-polygon-lookup", "knn-search", "bbox-filtering"]
RQ1_CONFIGS = ["local", "postgis", "duckdb"]

style = StyleConfig()
style.apply_rcparams()

## Data Loading

In [ ]:
# Persistent DuckDB file at cache/bench-cache.duckdb caches each remote read,
# keyed by RUN_ID. First run for a RUN_ID fetches from Azure (slow); later runs
# read the local table (fast). Samples are one tiny parquet per iteration
# (hundreds of thousands of blobs), so load_samples_cached fills the cache one
# (query_id, benchmark_run) cell at a time to keep peak memory flat -- the first
# fill is slow but resumable, later runs read the finished cache in one shot. Set
# REFRESH = True to force a re-fetch when data changed under the same RUN_ID.
REFRESH = False

db = open_cache()
db.execute(f"""
    CREATE OR REPLACE SECRET azure_secret(
        TYPE azure, PROVIDER config, ACCOUNT_NAME '{BLOB_STORAGE_ACCOUNT_NAME}'
    );
""")

experiments = load_experiments(BENCHMARKS_YML_PATH)
samples_df = load_samples_cached(db, BENCHMARK_CONTAINER, RUN_ID, refresh=REFRESH)
metadata_df = load_cached(
    db, "metadata", RUN_ID,
    lambda: load_metadata(db, METADATA_CONTAINER, BENCHMARK_METADATA_BLOB_NAME, RUN_ID),
    refresh=REFRESH,
)
costs_df = load_cached(
    db, "costs", RUN_ID,
    lambda: load_costs(db, BENCHMARK_CONTAINER, RUN_ID), refresh=REFRESH,
)
costs_df = enrich_costs(costs_df, experiments, WORKLOAD_TYPES)
samples_df = enrich_samples(samples_df, experiments, WORKLOAD_TYPES, ITERATION_CEILINGS)
successful_all = samples_df[samples_df["status"] == "success"].copy()

print(f"Loaded {len(samples_df)} samples, {len(successful_all)} successful")
print(f"Workloads: {sorted(samples_df['workload_type'].unique())}")
print(f"Dataset sizes: {sorted(samples_df['dataset_size'].unique())}")
print(f"Cost records: {len(costs_df)}")

## Validation

In [ ]:
run_all_validations(samples_df, successful_all, metadata_df, experiments)

In [ ]:
successful = successful_all[successful_all["workload_type"].isin(RQ1_WORKLOADS)].copy()
samples_rq1 = samples_df[samples_df["workload_type"].isin(RQ1_WORKLOADS)].copy()

print(f"RQ1 data: {len(successful)} successful iterations")
print(f"Configurations: {sorted(successful['configuration'].unique())}")
print(f"Dataset sizes: {sorted(successful['dataset_size'].unique())}")

## Statistical Analysis

In [ ]:
all_metrics = PRIMARY_METRICS + AUXILIARY_METRICS

table1 = build_descriptive_table(successful, all_metrics)
print(f"Table 1 (descriptive): {len(table1)} rows")
table1.head(10)

In [ ]:
table2 = build_cross_pass_table(table1)
print(f"Table 2 (cross-pass): {len(table2)} rows")
if len(table2) > 0:
    inconsistent = table2[~table2["consistent"]]
    print(f"  {len(inconsistent)} inconsistent (EXPLORATORY)")
table2.head(10)

In [ ]:
table3 = build_pairwise_table(successful, PRIMARY_METRICS)
print(f"Table 3 (pairwise): {len(table3)} rows")
table3.head(10)

In [ ]:
table4 = build_consistency_table(table3)
if len(table4) > 0:
    fc = table4["fully_consistent"].sum()
    total = len(table4)
    print(f"Table 4 (consistency): {total} comparisons, {fc} fully consistent ({fc / total:.0%})")
table4.head(10)

## RQ1 — Cloud-native vs. Traditional

In [ ]:
rq1_comparisons = table3.reset_index()
rq1_comparisons = rq1_comparisons[rq1_comparisons["workload_type"].isin(RQ1_WORKLOADS)]

rq1_consistency = table4.reset_index()
rq1_consistency = rq1_consistency[rq1_consistency["workload_type"].isin(RQ1_WORKLOADS)]

rq1_ranking = build_rq1_ranking(rq1_comparisons, rq1_consistency, table1, table2)
print("RQ1 Ranking Table:")
rq1_ranking

## RQ3 — Consistency Across Patterns

In [ ]:
rq3_data = rq1_consistency[rq1_consistency["metric"] == "elapsed_time"].copy()

if len(rq3_data) > 0:
    rq3_pivot = rq3_data.pivot_table(
        index=["config_a", "config_b"],
        columns="workload_type",
        values="direction_consistent",
        aggfunc="all",
    )
    print("RQ3 Cross-workload agreement (elapsed_time, direction consistency):")
    display(rq3_pivot)
else:
    print("No RQ1 consistency data available for RQ3 analysis")

## Visualisation

In [ ]:
plot_coverage_heatmap(
    samples_df, successful_all, style, FIGURES_DIR,
    metadata_df=metadata_df, experiments=experiments,
)

In [ ]:
plot_median_bars(successful, PRIMARY_METRICS, style, FIGURES_DIR, all_configs=RQ1_CONFIGS)

In [ ]:
plot_violin(successful, style, FIGURES_DIR, all_configs=RQ1_CONFIGS)

In [ ]:
plot_cpu_breakdown(successful, style, FIGURES_DIR, all_configs=RQ1_CONFIGS)

In [ ]:
plot_network_io(successful, style, FIGURES_DIR, all_configs=RQ1_CONFIGS)

In [ ]:
plot_cross_workload(successful, RQ1_WORKLOADS, RQ1_CONFIGS, style, FIGURES_DIR)

In [ ]:
plot_cv_iqr(successful, PRIMARY_METRICS, AUXILIARY_METRICS, style, FIGURES_DIR)

## Cost Analysis

In [ ]:
rq1_costs = costs_df[costs_df["workload_type"].isin(RQ1_WORKLOADS)].copy()

cost_table = build_cost_summary(rq1_costs)
if len(cost_table) > 0:
    print(f"Cost summary: {len(cost_table)} configuration×size cells")
    display(cost_table)
else:
    print("No cost data available for RQ1 workloads")

In [ ]:
plot_cost_bars(cost_table, style, FIGURES_DIR)

In [ ]:
plot_cost_breakdown(cost_table, style, FIGURES_DIR)

In [ ]:
plot_cost_performance(cost_table, successful, style, FIGURES_DIR)

## Cross-dataset-size Comparison

In [ ]:
plot_size_scaling(successful, style, FIGURES_DIR)

table5 = build_scaling_table(successful, style.size_order)
if len(table5) > 0:
    print("Scaling factors:")
    display(table5)

In [ ]:
plot_ranking_stability(successful, table3, style)

## LaTeX Export

In [ ]:
# Compact descriptive stats for main text
compact_t1 = compact_descriptive_table(table1)
compact_t1.to_latex(TABLES_DIR / "table1_descriptive_stats.tex", escape=False)

# Full descriptive stats for appendix
table1.to_latex(TABLES_DIR / "table1_descriptive_stats_full.tex", escape=True, float_format="%.4f")

table2.to_latex(TABLES_DIR / "table2_cross_pass.tex", escape=True, float_format="%.4f")

if len(table3) > 0:
    split_pairwise_effects(table3).to_latex(
        TABLES_DIR / "table3a_effect_sizes.tex", escape=False, float_format="%.4f"
    )
    split_pairwise_significance(table3).to_latex(
        TABLES_DIR / "table3b_significance.tex", escape=False, float_format="%.4f"
    )

if len(table4) > 0:
    format_consistency_table(table4).to_latex(
        TABLES_DIR / "table4_consistency.tex", escape=True, index=False
    )

if len(rq1_ranking) > 0:
    format_ranking_table(rq1_ranking).to_latex(
        TABLES_DIR / "rq1_ranking.tex", escape=True, float_format="%.4f"
    )

if len(table5) > 0:
    table5.to_latex(TABLES_DIR / "table5_size_scaling.tex", escape=True, float_format="%.4f")

if len(cost_table) > 0:
    format_cost_table(cost_table).to_latex(
        TABLES_DIR / "rq1_cost_summary.tex", escape=False,
    )

print(f"LaTeX tables exported to {TABLES_DIR}/")
for f in sorted(TABLES_DIR.glob("*.tex")):
    print(f"  {f.name}")